# 线性回归：从零实现

改编自 `d2l-ai/d2l-zh` 的 `chapter_linear-networks/linear-regression-scratch.md`，固定提交 `e6b18cc`，许可证 Apache-2.0。

本 Notebook 去除了 D2L 辅助包依赖，保留数据生成、小批量读取、损失函数、SGD 和训练流程。

In [1]:
import random

import torch

torch.manual_seed(42)
random.seed(42)

In [2]:
def synthetic_data(w, b, num_examples):
    x = torch.normal(0, 1, (num_examples, len(w)))
    y = x @ w + b
    y += torch.normal(0, 0.01, y.shape)
    return x, y.reshape((-1, 1))

true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)
features[0], labels[0]

(tensor([1.9269, 1.4873]), tensor([2.9871]))

In [3]:
def data_iter(batch_size, features, labels):
    indices = list(range(len(features)))
    random.shuffle(indices)
    for i in range(0, len(features), batch_size):
        batch_indices = torch.tensor(indices[i:min(i + batch_size, len(features))])
        yield features[batch_indices], labels[batch_indices]

x_batch, y_batch = next(data_iter(10, features, labels))
x_batch.shape, y_batch.shape

(torch.Size([10, 2]), torch.Size([10, 1]))

In [4]:
w = torch.normal(0, 0.01, size=(2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

def linear_regression(x, w, b):
    return x @ w + b

def squared_loss(y_hat, y):
    return (y_hat - y.reshape(y_hat.shape)) ** 2 / 2

def sgd(params, lr, batch_size):
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [5]:
learning_rate = 0.03
num_epochs = 3
batch_size = 10

for epoch in range(num_epochs):
    for x, y in data_iter(batch_size, features, labels):
        loss = squared_loss(linear_regression(x, w, b), y)
        loss.sum().backward()
        sgd([w, b], learning_rate, batch_size)
    with torch.no_grad():
        train_loss = squared_loss(linear_regression(features, w, b), labels).mean()
    print(f"epoch {epoch + 1}, loss {float(train_loss):.6f}")

epoch 1, loss 0.047187
epoch 2, loss 0.000199
epoch 3, loss 0.000050


In [6]:
with torch.no_grad():
    final_loss = squared_loss(linear_regression(features, w, b), labels).mean()
    w_error = torch.abs(true_w - w.reshape(true_w.shape)).max()
    b_error = torch.abs(true_b - b).max()

print("learned w:", w.reshape(-1).tolist())
print("learned b:", float(b.detach()))
print("max w error:", float(w_error))
print("b error:", float(b_error))

assert final_loss < 0.01
assert w_error < 0.1
assert b_error < 0.1
print("checks passed")

learned w: [1.9994313716888428, -3.399848699569702]
learned b: 4.199854850769043
max w error: 0.0005686283111572266
b error: 0.00014495849609375
checks passed
